# TrOCR-base Training — Phase 1 + Phase 2 freeform trace

Fine-tune **TrOCR-base** (`microsoft/trocr-base-stage1`, ~385M params) on Ogham, to compare
against the TrOCR-small (62M) result. Earlier runs at higher LRs either diverged outright
(LR 5e-5) or learnt only sluggishly with empty-prediction periods (LR 1e-5). This run uses
an even lower LR with more epochs to address Josh's suggestion that for a model of this
size relative to dataset size, lower learning rates may be necessary.

**Phase 1**: 200k clean synthetic, 15 epochs, LR 5e-6.

**Phase 2**: fine-tune on the 280-sample synthetic-freeform train split (seed 42), evaluate
on the same 35-sample freeform test split used by TrOCR-small / PARSeq / CNN+RNN.

Writes checkpoints to `checkpoints/base_unfrozen_lr5e6/` (Phase 1) and
`checkpoints/base_phase2_lr5e6/` (Phase 2) on Drive — the new experiment names keep this
run isolated from the previous LR 1e-5 attempt. Saves aggregated numbers to
`docs/base_phase2_results.json` in the repo.

---
## 1. Setup

In [ ]:
# Check GPU — base needs A100 (or 2× T4 with grad accum) for reasonable throughput.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Install deps (same pins as notebook 04)
!pip install -q transformers accelerate Pillow editdistance tqdm albumentations opencv-python-headless datasets huggingface_hub

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ogham_ocr'
for subdir in ['checkpoints', 'logs', 'experiments']:
    os.makedirs(f'{DRIVE_ROOT}/{subdir}', exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# Clone / pull repo
import os
REPO_DIR = '/content/ogham_ocr'

if os.path.exists(REPO_DIR):
    print('Repository already cloned, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/newsyd04/ogham-masters.git {REPO_DIR}

%cd {REPO_DIR}

import sys
sys.path.insert(0, REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# HF Hub login (for DaraTraining/ogham-synthetic-200k)
from huggingface_hub import login
login()

In [ ]:
# Copy fonts from Drive to local disk
from pathlib import Path
import shutil

DRIVE_FONT_DIR = Path(f'{DRIVE_ROOT}/datasets/fonts')
LOCAL_FONT_DIR = Path('/content/fonts')
REPO_FONT_DIR = Path(f'{REPO_DIR}/data/fonts')

LOCAL_FONT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_FONT_DIR.mkdir(parents=True, exist_ok=True)

source_dir = DRIVE_FONT_DIR if list(DRIVE_FONT_DIR.glob('*.ttf')) + list(DRIVE_FONT_DIR.glob('*.otf')) else REPO_FONT_DIR
for f in list(source_dir.glob('*.ttf')) + list(source_dir.glob('*.otf')):
    shutil.copy2(f, LOCAL_FONT_DIR / f.name)

FONT_DIR_STR = str(LOCAL_FONT_DIR)
print(f'Fonts in {FONT_DIR_STR}:')
for f in LOCAL_FONT_DIR.glob('*'):
    print(f'  {f.name}')

---
## 2. Configuration — TrOCR-base, lower-LR retry

Hyperparameter history for TrOCR-base on this corpus:

| Run | LR | Epochs | Outcome |
|---|---|---|---|
| Initial (small-model HPs) | 5e-5 | 3-7 | Diverged. CER climbed to 99-110%. |
| First corrected | 1e-5 | 10 | Loss decreased but CER 99.54% at epoch 1, slow convergence. |
| **This run** | **5e-6** | **15** | Address Josh's suggestion: lower LR for the model-vs-data ratio. |

Reasoning: TrOCR-base has 6× more parameters than TrOCR-small and the standard
rule-of-thumb LR for fine-tuning a large pretrained model on a moderate dataset is in the
1e-6 to 1e-5 range. The previous 1e-5 attempt was at the high end of that band and was
still learning slowly. 5e-6 sits more comfortably in the recommended range, at the cost
of needing more epochs to converge — hence the bump to 15.

Other settings unchanged from the previous attempt: physical batch 8, gradient
accumulation 4 (effective batch 32), encoder unfrozen from epoch 1, Ogham Unicode output.

In [ ]:
# ============================================================
# EXPERIMENT CONFIGURATION
# ============================================================
import os

DRIVE_ROOT = '/content/drive/MyDrive/ogham_ocr'
HF_DATASET_NAME = 'DaraTraining/ogham-synthetic-200k'

MODEL_SIZE = 'base'
P1_MODE = 'ogham'                 # Ogham-Unicode output
INIT_STRATEGY = 'random'          # matches small's winning config
P1_EPOCHS = 15                    # was 10, now longer to give lower LR room to converge
P1_BATCH_SIZE = 8
P1_LR = 5e-6                      # was 1e-5, now half — Josh's suggestion
P1_FREEZE_ENCODER = 0
P1_GRAD_ACCUM = 4                 # effective batch = 8 × 4 = 32
EXPERIMENT = 'base_unfrozen_lr5e6'  # isolates checkpoints from the previous LR 1e-5 run

CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/{EXPERIMENT}'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'Phase 1 will write to: {CHECKPOINT_DIR}')
print(f'LR: {P1_LR}')
print(f'Epochs: {P1_EPOCHS}')
print(f'Effective batch size: {P1_BATCH_SIZE * P1_GRAD_ACCUM}')

---
## 3. Phase 1 — 200k clean synthetic

Runs the repo's `train_colab.py` with TrOCR-base at LR 5e-6 for 15 epochs. Dataset streams
from HF Hub; cached to local NVMe on first access (~5 min).

**Time estimate**: ~6–9h on A100. Each epoch took ~2h at the previous settings, so 15 epochs
is a ~30h ceiling but will likely finish faster as later epochs are quicker without
encoder-warmup overhead. Checkpoints save to Drive after every improvement so disconnects
are safe (re-run this cell and it'll resume via `--resume`).

**Convergence expectations**:
- Epoch 1: CER likely > 90%, possibly empty predictions (normal for fresh Ogham vocab)
- Epochs 2–4: CER should drop into 30–60% if learning is healthy
- Epochs 5–9: target 5–15%
- Epochs 10–15: target 0.5–2%

**When to abort**: if by epoch 4 CER is still > 80%, the LR is too low — kill the run and
try LR 2e-5 instead. If val loss starts rising while train loss falls, that's overfitting
(the failure mode that killed the LR 5e-5 run); stop and keep the last best checkpoint.

In [ ]:
# Phase 1 training
!python scripts/train_colab.py \
    --mode {P1_MODE} \
    --model-size {MODEL_SIZE} \
    --phase 1 \
    --hf-dataset {HF_DATASET_NAME} \
    --font-dir {FONT_DIR_STR} \
    --epochs {P1_EPOCHS} \
    --batch-size {P1_BATCH_SIZE} \
    --lr {P1_LR} \
    --freeze-encoder-epochs {P1_FREEZE_ENCODER} \
    --gradient-accumulation {P1_GRAD_ACCUM} \
    --init-strategy {INIT_STRATEGY} \
    --experiment {EXPERIMENT} \
    --checkpoint-dir {CHECKPOINT_DIR}

In [ ]:
# Phase 1 training — RESUME (uncomment after a disconnect, then run this cell)
# !python scripts/train_colab.py \
#     --mode {P1_MODE} \
#     --model-size {MODEL_SIZE} \
#     --phase 1 \
#     --hf-dataset {HF_DATASET_NAME} \
#     --font-dir {FONT_DIR_STR} \
#     --epochs {P1_EPOCHS} \
#     --batch-size {P1_BATCH_SIZE} \
#     --lr {P1_LR} \
#     --freeze-encoder-epochs {P1_FREEZE_ENCODER} \
#     --gradient-accumulation {P1_GRAD_ACCUM} \
#     --init-strategy {INIT_STRATEGY} \
#     --experiment {EXPERIMENT} \
#     --checkpoint-dir {CHECKPOINT_DIR} \
#     --resume

In [ ]:
# Inspect Phase 1 history
import json
from pathlib import Path

history_path = Path(CHECKPOINT_DIR) / f'history_{P1_MODE}.json'
if history_path.exists():
    with open(history_path) as f:
        h = json.load(f)
    best_cer = min(h['val_cer'])
    best_epoch = h['val_cer'].index(best_cer) + 1
    print(f'Phase 1 best CER: {best_cer:.4f} at epoch {best_epoch}')
    print(f'Trained epochs: {len(h["val_cer"])}')
    print(f'Final train loss: {h["train_loss"][-1]:.4f}')
    print(f'Final val loss: {h["val_loss"][-1]:.4f}')
    # Quick visual: per-epoch trajectory
    print(f'\nVal CER per epoch:')
    for i, c in enumerate(h['val_cer'], 1):
        print(f'  Epoch {i:02d}: {c:.4f}')
else:
    print(f'No history at {history_path}')

---
## 4. Phase 2 — freeform trace fine-tune

Fine-tune the Phase 1 base checkpoint on 280 synthetic-freeform samples, held-out
35-sample test split (seed 42 — identical to the TrOCR-small / PARSeq / CNN+RNN splits
used in [docs/phase2_comparison.json](docs/phase2_comparison.json)).

Implemented inline (TrOCR-small's Phase 2 was done ad-hoc and never committed to a notebook).

In [ ]:
# ============================================================
# PHASE 2: Load freeform trace data + deterministic seed-42 split
# ============================================================
import csv, random, shutil
from pathlib import Path

FREEFORM_DRIVE = Path(f'{DRIVE_ROOT}/datasets/synthetic_freeform')
LOCAL_FF = Path(REPO_DIR) / 'ogham_dataset' / 'synthetic_freeform'
P2_CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/base_phase2_lr5e6'
os.makedirs(P2_CHECKPOINT_DIR, exist_ok=True)

P2_EPOCHS = 8
P2_LR = 2e-6                      # half of Phase 1 (which is half of previous), keeps the
                                  # adaptation gentle for an already-converged checkpoint
P2_BATCH_SIZE = 8
P2_GRAD_ACCUM = 4

LOCAL_FF.mkdir(parents=True, exist_ok=True)
!rsync -a {FREEFORM_DRIVE}/ {LOCAL_FF}/

ff_labels = {}
all_ids = []
with open(LOCAL_FF / 'labels.csv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        idx = int(row['image_file'].replace('freeform_', '').replace('.png', ''))
        ff_labels[idx] = (row['image_file'], row['ogham_text'])
        all_ids.append(idx)

rng = random.Random(42)
all_ids.sort()
rng.shuffle(all_ids)
n = len(all_ids)
n_train = int(n * 0.8); n_val = int(n * 0.1)
p2_train_ids = all_ids[:n_train]
p2_val_ids = all_ids[n_train:n_train + n_val]
p2_test_ids = all_ids[n_train + n_val:]

print(f'Freeform splits — train: {len(p2_train_ids)}, val: {len(p2_val_ids)}, test: {len(p2_test_ids)}')
print(f'Phase 2 LR: {P2_LR}')

In [ ]:
# ============================================================
# PHASE 2: Dataset + DataLoader (TrOCR-compatible)
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from src.training.tokenizer_extension import extend_tokenizer_with_ogham, resize_model_embeddings

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class FreeformTrOCRDataset(Dataset):
    def __init__(self, ids, labels, processor, tokenizer, img_dir, max_length=64):
        self.ids = ids
        self.labels = labels
        self.processor = processor
        self.tokenizer = tokenizer
        self.img_dir = Path(img_dir)
        self.max_length = max_length

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        idx = self.ids[i]
        fname, text = self.labels[idx]
        img = Image.open(self.img_dir / fname).convert('RGB')
        pixel_values = self.processor(img, return_tensors='pt').pixel_values.squeeze(0)
        enc = self.tokenizer(text, padding='max_length', truncation=True,
                             max_length=self.max_length, return_tensors='pt')
        labels = enc.input_ids.squeeze(0)
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels, 'text': text}

# Load Phase 1 best model
p1_best = Path(CHECKPOINT_DIR) / f'best_{P1_MODE}'
print(f'Loading Phase 1 best from {p1_best}')
processor = TrOCRProcessor.from_pretrained(str(p1_best))
model = VisionEncoderDecoderModel.from_pretrained(str(p1_best)).to(device)
tokenizer = processor.tokenizer

IMG_DIR = LOCAL_FF / 'images'
p2_train_ds = FreeformTrOCRDataset(p2_train_ids, ff_labels, processor, tokenizer, IMG_DIR)
p2_val_ds   = FreeformTrOCRDataset(p2_val_ids,   ff_labels, processor, tokenizer, IMG_DIR)
p2_test_ds  = FreeformTrOCRDataset(p2_test_ids,  ff_labels, processor, tokenizer, IMG_DIR)

p2_train_loader = DataLoader(p2_train_ds, batch_size=P2_BATCH_SIZE, shuffle=True, num_workers=2)
p2_val_loader   = DataLoader(p2_val_ds,   batch_size=P2_BATCH_SIZE, shuffle=False, num_workers=2)
p2_test_loader  = DataLoader(p2_test_ds,  batch_size=P2_BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
# ============================================================
# PRE-PHASE-2 BASELINE — Phase 1 model's performance on freeform test
# ============================================================
import editdistance

def compute_cer(preds, refs):
    total_chars = sum(len(r) for r in refs)
    if total_chars == 0: return 0.0
    return sum(editdistance.eval(p, r) for p, r in zip(preds, refs)) / total_chars

def eval_trocr(model, loader, tokenizer):
    model.eval()
    refs, preds = [], []
    with torch.no_grad():
        for batch in loader:
            pv = batch['pixel_values'].to(device)
            out = model.generate(pv, max_length=64)
            decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
            preds += [d.replace(' ', '') for d in decoded]
            refs  += [t.replace(' ', '') for t in batch['text']]
    cer = compute_cer(preds, refs)
    exact = sum(1 for p, r in zip(preds, refs) if p == r) / max(len(refs), 1)
    return cer, exact, refs, preds

pre_cer, pre_exact, _, _ = eval_trocr(model, p2_test_loader, tokenizer)
print(f'Pre-Phase-2 freeform test CER:   {pre_cer * 100:.2f}%')
print(f'Pre-Phase-2 freeform test exact: {pre_exact * 100:.2f}%')

In [ ]:
# ============================================================
# PHASE 2 TRAINING — fine-tune on freeform train split
# ============================================================
from torch.cuda.amp import autocast, GradScaler

optimizer = torch.optim.AdamW(model.parameters(), lr=P2_LR)
scaler = GradScaler()

best_val_cer = float('inf')
p2_best_path = Path(P2_CHECKPOINT_DIR) / 'best_base'
p2_history = {'train_loss': [], 'val_cer': [], 'val_exact': []}

for epoch in range(1, P2_EPOCHS + 1):
    model.train()
    total_loss, nb = 0.0, 0
    optimizer.zero_grad()
    for step, batch in enumerate(p2_train_loader):
        pv = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        with autocast():
            out = model(pixel_values=pv, labels=labels)
            loss = out.loss / P2_GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % P2_GRAD_ACCUM == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        total_loss += loss.item() * P2_GRAD_ACCUM
        nb += 1

    train_loss = total_loss / max(nb, 1)
    val_cer, val_exact, _, _ = eval_trocr(model, p2_val_loader, tokenizer)
    p2_history['train_loss'].append(train_loss)
    p2_history['val_cer'].append(val_cer * 100)
    p2_history['val_exact'].append(val_exact * 100)

    print(f'Epoch {epoch:02d} | train_loss {train_loss:.4f} | val_cer {val_cer*100:.2f}% | val_exact {val_exact*100:.2f}%')

    if val_cer < best_val_cer:
        best_val_cer = val_cer
        model.save_pretrained(p2_best_path)
        processor.save_pretrained(p2_best_path)
        print(f'  New best — saved to {p2_best_path}')

print(f'\nPhase 2 best val CER: {best_val_cer * 100:.2f}%')

In [ ]:
# ============================================================
# PHASE 2 FINAL EVAL — on held-out freeform test split
# ============================================================
model = VisionEncoderDecoderModel.from_pretrained(p2_best_path).to(device)
post_cer, post_exact, refs, preds = eval_trocr(model, p2_test_loader, tokenizer)

print('=' * 70)
print(f'TrOCR-base Phase 2 — TEST SPLIT ({len(refs)} samples, seed 42)')
print('=' * 70)
print(f'Pre-P2  CER:   {pre_cer  * 100:6.2f}%   exact: {pre_exact  * 100:5.2f}%')
print(f'Post-P2 CER:   {post_cer * 100:6.2f}%   exact: {post_exact * 100:5.2f}%')
print(f'Lift:          {(pre_cer - post_cer) * 100:+.2f} percentage points')

print('\nSample predictions:')
for i in range(min(10, len(refs))):
    mark = '✓' if preds[i] == refs[i] else '✗'
    print(f'  {mark} ref: {refs[i]}')
    print(f'     pred: {preds[i]}')

In [ ]:
# ============================================================
# Save aggregated results to repo for plotting scripts
# ============================================================
import json

# Phase 1 best
p1_history = json.load(open(Path(CHECKPOINT_DIR) / f'history_{P1_MODE}.json'))
p1_best_cer = min(p1_history['val_cer'])
p1_best_idx = p1_history['val_cer'].index(p1_best_cer)
p1_best_exact = p1_history['val_exact'][p1_best_idx] if 'val_exact' in p1_history else None

out = {
    'model': 'trocr-base',
    'experiment': EXPERIMENT,
    'params_M': sum(p.numel() for p in model.parameters()) / 1e6,
    'phase1': {
        'lr': P1_LR,
        'batch_size': P1_BATCH_SIZE,
        'grad_accum': P1_GRAD_ACCUM,
        'epochs_run': len(p1_history['val_cer']),
        'best_cer_pct': p1_best_cer * 100 if p1_best_cer < 1 else p1_best_cer,
        'best_exact_pct': p1_best_exact,
    },
    'phase2': {
        'lr': P2_LR,
        'batch_size': P2_BATCH_SIZE,
        'grad_accum': P2_GRAD_ACCUM,
        'epochs_run': P2_EPOCHS,
        'pre_p2_freeform_cer_pct': pre_cer * 100,
        'post_p2_freeform_cer_pct': post_cer * 100,
        'pre_p2_freeform_exact_pct': pre_exact * 100,
        'post_p2_freeform_exact_pct': post_exact * 100,
        'lift_pp': (pre_cer - post_cer) * 100,
    },
    'test_split_size': len(refs),
    'seed': 42,
}

out_path = Path(REPO_DIR) / 'docs' / 'base_phase2_results.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print(f'Saved {out_path}')
print(json.dumps(out, indent=2, ensure_ascii=False))